In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "270_build_silver_fact_bid_project.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.silver.fact_bid_item"
DIM_PROJECT_TABLE = f"{catalog}.silver.dim_project"
DIM_PROJECT_CLASSIFICATION_TABLE = f"{catalog}.silver.dim_project_classification"
DIM_COUNTY_TABLE = f"{catalog}.silver.dim_county"
TARGET_TABLE = f"{catalog}.silver.fact_bid_project"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build silver.fact_bid_project
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    AS
    WITH bid_rollup AS (
        SELECT
              project_id
            , vendor_name_standardized

            , MIN(vendor_name) AS vendor_name
            , MIN(project_key) AS project_key
            , MIN(vendor_key) AS vendor_key

            , COUNT(*) AS bid_item_count
            , COUNT(DISTINCT item_specification_key) AS distinct_item_specification_count
            , COUNT(DISTINCT bid_item_sequence_number) AS distinct_bid_item_sequence_count
            , COUNT(DISTINCT bid_submit_sequence_number) AS distinct_submit_sequence_count

            , SUM(COALESCE(bid_item_quantity, 0)) AS total_bid_item_quantity
            , SUM(COALESCE(bid_item_quantity, 0) * COALESCE(bid_item_unit_price_amount, 0)) AS bid_item_extended_total
            , MIN(bid_total_amount) AS min_bid_total_amount
            , MAX(bid_total_amount) AS max_bid_total_amount

            , MIN(bid_rank_sequence_number) AS min_bid_rank_sequence_number
            , MAX(bid_rank_sequence_number) AS max_bid_rank_sequence_number

            , MAX(CASE WHEN low_bidder_flag THEN 1 ELSE 0 END) = 1 AS has_any_low_bidder_items

            , SUM(CASE WHEN is_payment_adjustment_item THEN 1 ELSE 0 END) AS payment_adjustment_item_count
            , SUM(CASE WHEN is_special_deduction_item THEN 1 ELSE 0 END) AS special_deduction_item_count
            , SUM(CASE WHEN is_nonstandard_adjustment_item THEN 1 ELSE 0 END) AS nonstandard_adjustment_item_count

            , MAX(CASE WHEN vendor_project_has_resubmission_history THEN 1 ELSE 0 END) = 1 AS vendor_project_has_resubmission_history
            , MAX(CASE WHEN item_has_resubmission_history THEN 1 ELSE 0 END) = 1 AS any_item_has_resubmission_history
            , MAX(CASE WHEN item_changed_across_submit_sequences THEN 1 ELSE 0 END) = 1 AS any_item_changed_across_submit_sequences
            , MAX(CASE WHEN item_missing_from_latest_project_submit THEN 1 ELSE 0 END) = 1 AS any_item_missing_from_latest_project_submit
            , MAX(CASE WHEN item_changed_value_across_submits THEN 1 ELSE 0 END) = 1 AS any_item_changed_value_across_submits

            , MAX(source_updated_at) AS latest_source_updated_at
            , MAX(ingested_at_utc) AS latest_ingested_at_utc
        FROM {SOURCE_TABLE}
        GROUP BY
              project_id
            , vendor_name_standardized
    )

    , ranked_bids AS (
        SELECT
              *
            , MIN(max_bid_total_amount) OVER (PARTITION BY project_id) AS lowest_bid_amount_in_project
            , MAX(max_bid_total_amount) OVER (PARTITION BY project_id) AS highest_bid_amount_in_project
            , COUNT(*) OVER (PARTITION BY project_id) AS bidder_count_in_project
        FROM bid_rollup
    )

    , final_ranked AS (
        SELECT
              *
            , CASE
                  WHEN max_bid_total_amount = lowest_bid_amount_in_project THEN TRUE
                  ELSE FALSE
              END AS is_low_bidder

            , ROW_NUMBER() OVER (
                PARTITION BY project_id
                ORDER BY
                      max_bid_total_amount ASC NULLS LAST
                    , min_bid_rank_sequence_number ASC NULLS LAST
                    , vendor_name_standardized
              ) AS project_bid_rank
        FROM ranked_bids
    )

    SELECT
          md5(concat_ws('|', COALESCE(r.project_id, ''), COALESCE(r.vendor_name_standardized, ''))) AS bid_project_fact_key

        , r.project_key
        , r.vendor_key

        , r.project_id
        , r.vendor_name
        , r.vendor_name_standardized

        , pc.project_classification_key
        , dc.county_key

        , dp.project_name
        , dp.project_number
        , dp.state_project_number
        , dp.control_section_job_csj
        , dp.controlling_project_id_ccsj
        , dp.project_type
        , dp.project_classification
        , dp.project_actual_let_date
        , dp.project_estimated_let_date
        , dp.project_limits_from
        , dp.project_limits_to
        , dp.county
        , dc.county_location AS location
        , dp.district_division
        , dp.highway
        , dp.short_description
        , dp.federal_project_number

        , r.bid_item_count
        , r.distinct_item_specification_count
        , r.distinct_bid_item_sequence_count
        , r.distinct_submit_sequence_count
        , r.total_bid_item_quantity
        , r.bid_item_extended_total
        , r.min_bid_total_amount
        , r.max_bid_total_amount

        , CASE
              WHEN r.min_bid_total_amount IS NOT NULL
               AND r.max_bid_total_amount IS NOT NULL
               AND r.min_bid_total_amount <> r.max_bid_total_amount
              THEN TRUE
              ELSE FALSE
          END AS has_conflicting_bid_total_amounts

        , CASE
              WHEN r.max_bid_total_amount IS NOT NULL
               AND r.bid_item_extended_total IS NOT NULL
              THEN ABS(r.max_bid_total_amount - r.bid_item_extended_total)
              ELSE NULL
          END AS bid_total_vs_item_rollup_abs_diff

        , r.min_bid_rank_sequence_number
        , r.max_bid_rank_sequence_number
        , r.has_any_low_bidder_items
        , r.is_low_bidder

        , r.payment_adjustment_item_count
        , r.special_deduction_item_count
        , r.nonstandard_adjustment_item_count

        , r.vendor_project_has_resubmission_history
        , r.any_item_has_resubmission_history
        , r.any_item_changed_across_submit_sequences
        , r.any_item_missing_from_latest_project_submit
        , r.any_item_changed_value_across_submits

        , r.project_bid_rank
        , r.bidder_count_in_project
        , r.lowest_bid_amount_in_project
        , r.highest_bid_amount_in_project

        , CASE
              WHEN r.highest_bid_amount_in_project IS NOT NULL
               AND r.lowest_bid_amount_in_project IS NOT NULL
              THEN r.highest_bid_amount_in_project - r.lowest_bid_amount_in_project
              ELSE NULL
          END AS bid_spread_amount_in_project

        , CASE
              WHEN r.lowest_bid_amount_in_project IS NOT NULL
               AND r.lowest_bid_amount_in_project <> 0
               AND r.max_bid_total_amount IS NOT NULL
              THEN r.max_bid_total_amount / r.lowest_bid_amount_in_project
              ELSE NULL
          END AS bid_vs_low_bid_ratio

        , r.latest_source_updated_at
        , r.latest_ingested_at_utc

    FROM final_ranked r
    LEFT JOIN {DIM_PROJECT_TABLE} dp
        ON r.project_id = dp.project_id
    LEFT JOIN {DIM_PROJECT_CLASSIFICATION_TABLE} pc
        ON (
            CASE
                WHEN COALESCE(dp.project_classification, '') = '' THEN 'UNKNOWN'
                ELSE dp.project_classification
            END
        ) = pc.project_classification
    LEFT JOIN {DIM_COUNTY_TABLE} dc
        ON (
            CASE
                WHEN COALESCE(dp.county, '') = '' THEN 'UNKNOWN'
                WHEN dp.county = 'De Witt' THEN 'DeWitt'
                ELSE dp.county
            END
        ) = dc.county
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Project/vendor-level bid fact table built from silver.fact_bid_item.'
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
    """).collect()[0]["row_count"]

    missing_project_classification_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE project_classification_key IS NULL
    """).collect()[0]["missing_count"]

    missing_county_keys = spark.sql(f"""
        SELECT COUNT(*) AS missing_count
        FROM {TARGET_TABLE}
        WHERE county_key IS NULL
    """).collect()[0]["missing_count"]

    log_run(
        "SUCCESS",
        row_count,
        "Built silver.fact_bid_project successfully."
    )

    print("=" * 70)
    print("Built silver.fact_bid_project")
    print(f"Row count: {row_count:,}")
    print("=" * 70)
    print("Validation:")
    print(f"  missing_project_classification_keys: {missing_project_classification_keys:,}")
    print(f"  missing_county_keys: {missing_county_keys:,}")

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise